# Project 18: Limited-Angle CT Reconstruction

This demo first displays included ideal phantoms (CPU). Later cells require your ASTRA-generated data and a trained checkpoint. No results or weights are precomputed. Launch Jupyter from the project root or the notebooks folder.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'limited_ct').exists():
    ROOT = ROOT.parent
assert (ROOT / 'limited_ct').exists(), 'Open from the project root or notebooks folder'
sys.path.insert(0, str(ROOT))
from limited_ct.config import read_config, geometry_metadata
from limited_ct.phantoms import split_plan
cfg = read_config(ROOT / 'configs/project18.json')
cfg['samples_per_condition'] = 1000
print('Local training: 1,000 existing phantom images, 2,000 angle-specific pairs.')
print('Data backend: CPU fan-beam FBP. Model training device is selected separately.')


## Included raster examples — these are not reconstruction results

In [ ]:
files = sorted((ROOT / 'examples/phantoms').glob('*.npy'))
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, path in zip(axes.flat, files):
    ax.imshow(np.load(path), cmap='gray', vmin=0, vmax=1)
    ax.set_title(path.stem)
    ax.axis('off')
plt.tight_layout()


## Verify the grouped split and view counts

In [ ]:
from collections import Counter
from limited_ct.config import indices_for_range
plan = split_plan(cfg)
print('Samples per condition:', Counter(r['split'] for r in plan))
for angle in (90, 120):
    print(angle, 'degrees:', len(indices_for_range(cfg, angle)), 'measured views')


## Import the existing dataset and train locally

Place both training ZIP parts in `datasets/`. From the project root run:

```bash
python run_local_cuda.py --prepare-only
python run_local_cuda.py
```

Neural training uses the local CUDA GPU. Existing FBP images retain their CPU
fan-beam generation provenance. SART and TV-SART are not present in these ZIPs.
The later cells use validation examples and your trained checkpoint.


In [ ]:
DATA = ROOT / 'data/limited_angle_ct_1000'
ANGLE = 90
SPLIT = 'val'  # use validation while exploring; final test belongs in the fixed protocol
assert (DATA / 'manifest.json').exists(), 'Run python run_local_cuda.py --prepare-only to import both dataset ZIPs first'
manifest = json.loads((DATA / 'manifest.json').read_text())
record = next(r for r in manifest['records'] if r['angle'] == ANGLE and r['split'] == SPLIT)
folder, shared = DATA / record['path'], DATA / record['shared']
arrays = {
    'Ideal': np.load(shared / 'phantom.npy'),
    'Full FBP target': np.load(shared / 'target.npy'),
    'Limited FBP': np.load(folder / 'fbp.npy'),
}
fig, axes = plt.subplots(1, len(arrays), figsize=(4 * len(arrays), 4))
for ax, (name, image) in zip(axes, arrays.items()):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()


In [ ]:
if (folder / 'sinogram_padded.npy').is_file():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(np.load(folder / 'sinogram_padded.npy'), aspect='auto', cmap='gray')
    axes[0].set(title='Zero-padded line integrals', xlabel='Detector index', ylabel='Full-grid view index')
    axes[1].imshow(np.load(folder / 'measured_mask.npy'), aspect='auto', cmap='gray', vmin=0, vmax=1)
    axes[1].set(title='Measured=1; missing=0', xlabel='Detector index', ylabel='Full-grid view index')
    plt.tight_layout()
else:
    print('Sinogram panels are optional. The two training ZIPs contain FBP images only.')
    print('Add and prepare the optional measured-sinogram archive to view projection panels.')


## Correct one validation image and inspect metrics

In [ ]:
import torch
from limited_ct.evaluate import load_generator
from limited_ct.metrics import image_metrics, directional_error, directional_rmse
CHECKPOINT = ROOT / f'runs/local_cuda_1000/{ANGLE}_gan_adv1/best.pt'
assert CHECKPOINT.exists(), 'Train the generator before running this cell'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, checkpoint = load_generator(CHECKPOINT, device)
assert checkpoint['settings']['angle'] == ANGLE
x = np.load(folder / (checkpoint['settings']['input'] + '.npy'))
with torch.no_grad():
    prediction = model(torch.from_numpy(x)[None, None].to(device)).cpu().numpy()[0, 0]
prediction = np.clip(prediction, 0, 1)
target, ideal = arrays['Full FBP target'], arrays['Ideal']
print('Against full FBP:', image_metrics(prediction, target))
print('Against ideal:', image_metrics(prediction, ideal))
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, image) in zip(axes, {'Input': x, 'Corrected': prediction, 'Ideal': ideal}.items()):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1); ax.set_title(name); ax.axis('off')
axes[3].imshow(abs(prediction-ideal), cmap='magma', vmin=0, vmax=.3)
axes[3].set_title('|Error| vs ideal [0, .3]'); axes[3].axis('off')
plt.tight_layout()


In [ ]:
bins = cfg['orientation_bins']
sse, counts = directional_error(prediction, ideal, bins, cfg['edge_threshold_fraction'], ANGLE / 2)
theta = np.deg2rad((np.arange(bins) + .5) * 180 / bins)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(7, 5))
ax.plot(theta, directional_rmse(sse, counts))
ax.set_thetamin(0); ax.set_thetamax(180)
ax.set_title('Validation pixel RMSE at ideal edges\nNormal relative to source-angle center')
print('Edge counts by orientation:', counts)
plt.show()


## Interpretation

Compare the 90° and 120° conditions on matched phantom families. Investigate invented or missing boundaries even when a prediction looks sharper. Directional bins describe reference edge normals, not an exact position-dependent fan-beam visibility test. The CLI performs complete per-sample/per-family evaluation, pooled directional curves, and optional measured-ray reprojection checks. Do not tune methods using held-out test images.